In [1]:
import pandas as pd

In [2]:
df = pd.read_json('in/articles.jsonl', lines=True)
print(df.shape)
df.head()

(2922, 9)


,title,subtitle,from_publication,date_publication,json_ld,content,url,html_path,scraped_at
0,É #FATO: Cemitério na Jamaica tem 'totens' com...,Vídeo viral foi gravado no cemitério de Annott...,PorRedação g1,23/03/2026 15h51Atualizadohá 2 dias,None,"[{'tag': 'figcaption', 'text': '1 de 4 É #FATO...",https://g1.globo.com/fato-ou-fake/noticia/2026...,data/html_cache/000000_article.html,2026-03-25 21:05:34.287914
1,É #FAKE que TSE proibiu Bolsonaro de baixar pr...,"Ao Fato ou Fake, TSE disse: 'conteúdo é falso'...",PorRedação g1,25/03/2026 17h42Atualizadohá uma hora,"[{'articleSection': ['G1', 'Fato ou Fake']}]","[{'tag': 'figcaption', 'text': '1 de 3 TSE não...",https://g1.globo.com/fato-ou-fake/noticia/2026...,data/html_cache/000001_article.html,2026-03-25 21:05:35.186873
2,É #FAKE que 'Lei Felca' causará bloqueio de iP...,ECA Digital não prevê bloqueio de aparelhos ou...,"Por O Globo, Luiz Eduardo de Castro",25/03/2026 09h07Atualizadohá 11 horas,None,"[{'tag': 'figcaption', 'text': '1 de 3 É #FAKE...",https://g1.globo.com/fato-ou-fake/noticia/2026...,data/html_cache/000002_article.html,2026-03-25 21:05:35.566225
3,É #FAKE foto de posto com litro de gasolina co...,Posts enganosos utilizam foto com valores alte...,PorRedação g1,24/03/2026 15h47Atualizadohá um dia,None,"[{'tag': 'figcaption', 'text': '1 de 6 É #FAKE...",https://g1.globo.com/fato-ou-fake/noticia/2026...,data/html_cache/000003_article.html,2026-03-25 21:05:35.829453
4,É #FATO: Vídeo mostra canguru deitado recebend...,Cena foi gravada em maio de 2025 em local envo...,"PorMel Trench, g1",24/03/2026 10h01Atualizadohá 10 horas,None,"[{'tag': 'section', 'text': 'Assista também no...",https://g1.globo.com/fato-ou-fake/noticia/2026...,data/html_cache/000004_article.html,2026-03-25 21:05:36.082339


In [ ]:
from unidecode import unidecode

def extract_blockquote_and_next(url, content_list):
    results = []
    for i, item in enumerate(content_list):
        if not item.get('tag') or not item['tag'] == 'blockquote':
            continue

        # Try to find matching <p> within 2-3 elements away
        found_match = None
        for j in range(i + 1, min(i + 4, len(content_list))):  # i+1 to i+3 (2-3 elements)
            next_item = content_list[j]
            if not next_item.get('tag') or next_item['tag'] != 'p':
                continue
            
            text = next_item.get('text', '')
            unidecoded_text = unidecode(str(text)).strip().lower()
            lower_text = str(text).strip().lower()
            if (
                any(keyword in unidecoded_text for keyword in ['#fato', '#fake', '#naoebemassim'])
                and 'foto:' not in lower_text 
                and 'veja o que e #fato ou #fake sobre' not in unidecoded_text
                and
                ):
                found_match = next_item
                break
        
        # If no match within 2-3 elements, check if keywords are in any <p> in the document
        if found_match is None:
            for content_item in content_list:
                if not content_item.get('tag') or content_item['tag'] != 'p':
                    continue
                text = content_item.get('text', '')
                unidecoded_text = unidecode(str(text)).strip().lower()
                lower_text = str(text).strip().lower()
                if any(keyword in unidecoded_text for keyword in ['#fato', '#fake', '#naoebemassim']) and 'foto:' not in lower_text and 'veja o que e #fato ou #fake sobre' not in unidecoded_text:
                    found_match = content_item
                    break
        
        # Only append if we found a matching <p> tag
        if found_match is not None:
            results.append({
                'blockquote': item,
                'next_item': found_match,
                'url': url
            })
    
    return results

extracted = df[['url', 'content']].apply(lambda row: extract_blockquote_and_next(row['url'], row['content']), axis=1)
extracted_df = pd.DataFrame([item for sublist in extracted for item in sublist])

In [111]:
from unidecode import unidecode

def extract_blockquote_and_next(url, content_list):
    results = []
    for i, item in enumerate(content_list):
        if not item.get('tag') or not item['tag'] == 'blockquote':
            continue

        found_p = None
        found_p_text = None
        found_p_index = None
        
        # Try to find <p> within 2-3 elements away
        for j in range(i + 1, min(i + 4, len(content_list))):
            next_item = content_list[j]
            if next_item.get('tag') != 'p':
                continue
            
            text = next_item.get('text', '')
            unidecoded_text = unidecode(str(text)).strip().lower()
            lower_text = str(text).strip().lower()
            if (
                any(keyword in unidecoded_text for keyword in ['#fato', '#fake', '#naoebemassim'])
                and 'foto:' not in lower_text 
                and 'veja o que e #fato ou #fake sobre' not in unidecoded_text
            ):
                found_p = next_item
                found_p_text = text
                found_p_index = j
                break
        
        # If we found a <p>, check for <ul> in the next 1-3 elements
        if found_p is not None:
            final_text = found_p_text
            
            for k in range(found_p_index + 1, min(found_p_index + 4, len(content_list))):
                ul_item = content_list[k]
                if ul_item.get('tag') == 'ul':
                    li_texts = []
                    
                    # Check if <li> are children of <ul>
                    if 'children' in ul_item:
                        for li in ul_item.get('children', []):
                            if li.get('tag') == 'li':
                                li_texts.append(li.get('text', '').strip())
                    else:
                        # Otherwise, look for <li> items following the <ul>
                        for m in range(k + 1, len(content_list)):
                            li_item = content_list[m]
                            if li_item.get('tag') == 'li':
                                li_texts.append(li_item.get('text', '').strip())
                            else:
                                break  # Stop at first non-<li>
                    
                    if li_texts:
                        final_text = found_p_text + ' ' + ';'.join(li_texts)
                    break
            
            results.append({
                'blockquote': item,
                'next_item': found_p,
                'next_item_text': final_text,
                'url': url
            })
            continue
        
        # If no <p> match within 2-3 elements, check anywhere in document
        for content_item in content_list:
            if not content_item.get('tag') or content_item['tag'] != 'p':
                continue
            text = content_item.get('text', '')
            unidecoded_text = unidecode(str(text)).strip().lower()
            lower_text = str(text).strip().lower()
            if any(keyword in unidecoded_text for keyword in ['#fato', '#fake', '#naoebemassim']) and 'foto:' not in lower_text and 'veja o que e #fato ou #fake sobre' not in unidecoded_text:
                results.append({
                    'blockquote': item,
                    'next_item': content_item,
                    'next_item_text': text,
                    'url': url
                })
                break
    
    return results

extracted = df[['url', 'content']].apply(lambda row: extract_blockquote_and_next(row['url'], row['content']), axis=1)
extracted_df = pd.DataFrame([item for sublist in extracted for item in sublist])

In [112]:
print(extracted_df.shape)

(3751, 4)


In [113]:
# make the url appear completely on head view
pd.set_option('display.max_colwidth', None)

extracted_df['blockquote_text'] = extracted_df['blockquote'].apply(lambda x: x.get('text', ''))
extracted_df['next_item_text'] = extracted_df['next_item'].apply(lambda x: x.get('text', ''))

In [114]:
# keep only rows in which blockquote_text starts with '"'
#extracted_df = extracted_df[extracted_df['next_item_text'].str.contains('declaração|declaracao|declarou|declarou que|declarou ser|declarou ser que|afirmou|afirmacao|afirmação|afirmou que|afirmou ser|afirmou ser que', case=False, na=False)]
#print(extracted_df.shape)

In [115]:
extracted_df[['blockquote_text', 'next_item_text', 'url']].sample(10)

,blockquote_text,next_item_text,url
3688,"""O ministro Tarcísio e o presidente Bolsonaro sinalizaram esta vontade de empenhar este recurso da multa da Vale na implementação da linha 2 do metrô. (...) Não é uma coisa que vai acontecer se o Bruno Engler for prefeito. É uma coisa que o governo federal vai fazer, independente de quem ganhar. O prefeito eleito, se ele tiver disponibilidade para fazer este empreendimento em parceria com União, acredito que vai sair” (em entrevista ao jornal ‘Hoje em Dia’, em 6/10)","A declaração é #FAKE. Veja o porquê:Em um momento, o candidato cita um valor e diz que ele não foi enviado para a cidade. Em outro momento, reforça que a cidade não recebeu nenhuma ajuda. Não é verdade. De acordo com o Ministério do Desenvolvimento Regional, foram destinados R$ 10,8 milhões para ações de Defesa Civil em Belo Horizonte.",https://g1.globo.com/fato-ou-fake/noticia/2020/10/09/veja-o-que-e-fato-ou-fake-nas-declaracoes-dos-candidatos-a-prefeitura-de-belo-horizonte-na-2a-semana-de-campanha.ghtml
2618,"“Niterói, nos últimos oito anos da minha gestão, conquistou pela CGU, que é um órgão independente do governo federal, e também pelo Ministério Público Federal, o primeiro lugar em transparência na administração municipal dentre os 92 municípios do estado.”","A declaração é #FATO. Veja por quê:A Escala Brasil, levantamento da Controladoria Geral da União (CGU), avaliou 665 municípios no país, em 2020, e considerou Niterói a 1ª cidade no ranking no estado, ao lado das cidades de Mesquita, na Baixada Fluminense, e São Pedro da Aldeia, na Região dos Lagos.",https://g1.globo.com/rj/rio-de-janeiro/eleicoes/2022/noticia/2022/09/12/veja-o-que-e-fato-ou-fake-na-entrevista-de-rodrigo-neves-ao-rj1.ghtml
709,"""A Receita Federal não tem poder para criar impostos. Isso é uma atribuição exclusiva do Poder Legislativo – ou seja, só pode ser feito por meio de leis aprovadas pelo Congresso Nacional (ao nível federal), pelas Assembleias Legislativas (ao nível estadual) ou pelas Câmaras Municipais (em nível municipal). Este é outro pilar fundamental do sistema tributário brasileiro, o Princípio da Estrita Legalidade Tributária, que garante que nenhum tributo pode ser instituído ou majorado sem lei que o estabeleça"", diz Movan.","Circulam nas redes sociais publicações dizendo que aReceita Federalirá cobrar imposto de adultos que moram com os pais, a partir de 2026.É #FAKE.",https://g1.globo.com/fato-ou-fake/noticia/2025/09/25/e-fake-que-a-receita-federal-vai-cobrar-impostos-de-adultos-que-moram-com-os-pais.ghtml
2883,"""Esse governo (Jair Bolsonaro) cortou 98% da construção de casas populares para quem ganha até 1,5 salário mínimo"". (em entrevista a O Globo, em 25/08)","#NÃOÉBEMASSIM. Veja por quê:De acordo com a pesquisa Genial/Quaest (feita entre os dias 11 e 12 de agosto), os eleitores entrevistados que não apresentaram decisão clara, sem estímulo,correspondem a 36%. No entanto, segundo a última pesquisa com perguntas espontâneas do Instituto Datafolha (feita entre 16 e 18 de agosto), o número de indecisos no mês giraem torno de 22%, o que equivale a quase 1/4 dos entrevistados.",https://g1.globo.com/fato-ou-fake/eleicoes/noticia/2022/08/26/veja-o-que-e-fato-ou-fake-na-sabatina-de-simone-tebet-para-o-globo-valor-e-cbn.ghtml
1558,"""Ele [o produto] é classificado como suplemento alimentar, e suplementos não podem fazer alegações terapêuticas, como o tratamento de doenças, incluindo a disfunção erétil. De acordo com as Resoluções RDC Anvisa nº 27/2010 e nº 240/2018, suplementos alimentares estão isentos de registro na Anvisa, desde que cumpram os requisitos estabelecidos nessas normas"", explica Messina.","Circula nas redes sociais uma publicação que imita a página dog1com o título ""Revolucionário: sexóloga aprova spray de alga pra tratar disfunção erétil e ameaça quebrar indústria farmacêutica"".É #FAKE.",https://g1.globo.com/fato-ou-fake/noticia/2025/01/08/e-fake-que-spray-de-alga-trate-ou-previna-disfuncao-eretil.ghtml
1959,"“

In [116]:
# apply status map
# if naoebemassim in unidecode(str(next_item_text)).strip().lower() then status = 'NAOEBEMASSIM'
# elif 'fato' in next_item_text.upper() then status = 'FATO'
# elif 'fake' in next_item_text.upper() then status = 'FAKE'
# else status = 'UNKNOWN'

def determine_status(text):
    text_upper = text.upper()
    text_unidecode = unidecode(str(text)).strip().lower()
    
    if 'naoebemassim' in text_unidecode:
        return 'NAOEBEMASSIM'
    elif 'FATO' in text_upper:
        return 'FATO'
    elif 'FAKE' in text_upper:
        return 'FAKE'
    else:
        return 'UNKNOWN'
    
extracted_df['status'] = extracted_df['next_item_text'].apply(determine_status)
extracted_df[['blockquote_text', 'next_item_text', 'url', 'status']].head(50)

,blockquote_text,next_item_text,url,status
0,"O vídeo foi gravado no cemitério de Annotto Bay, na região nordeste do país. Segundo uma reportagem do jornal local ""Jamaica Observer"", os totens se tornaram uma tradição popular por lá, para auxiliar na localização dos túmulos e manter viva a memória dos falecidos. Um funcionário explicou, em entrevista, que os próprios familiares das pessoas enterradas escolhem as fotos, que depois são transformadas em grandes imagens em 2D. Para fixá-las, uma haste de aço é cimentada na sepultura.","Circula nas redes sociais um vídeo que mostra um cemitério na Jamaica repleto de totens de papelão, confeccionados em tamanho real, para representar as pessoas enterradas no local.É #FATO.",https://g1.globo.com/fato-ou-fake/noticia/2026/03/23/e-fato-cemiterio-na-jamaica-tem-totens-com-fotos-em-tamanho-real-de-pessoas-enterradas.ghtml,FATO
1,OFato ou Fakepesquisou pelo cemitério de Annotto Bay no Google Maps e confirmou que os totens são visíveis a partir de ruas ao lado do cemitério.Veja as imagens disponíveis no Google Street View:,"Circula nas redes sociais um vídeo que mostra um cemitério na Jamaica repleto de totens de papelão, confeccionados em tamanho real, para representar as pessoas enterradas no local.É #FATO.",https://g1.globo.com/fato-ou-fake/noticia/2026/03/23/e-fato-cemiterio-na-jamaica-tem-totens-com-fotos-em-tamanho-real-de-pessoas-enterradas.ghtml,FATO
2,"Mas isso não é verdade.OFato ou Fakeenviou o material à assessoria de imprensa do TSE, que respondeu, por e-mail:""O conteúdo é falso. Em 22 de março de 2022, o TSE arquivou consulta formulada [em 16 de fevereiro] pelaAdvocacia-Geral da União(AGU) que questionava se é possível, em ano eleitoral, a redução, a partir de lei aprovada pelo Congresso Nacional, da alíquota de impostos e contribuições sobre produtos e insumos. Por unanimidade, o Plenário decidiu não conhecer da consulta, o que significa que o julgamento não foi levado adiante para análise do mérito da questão, que envolve, entre outros pontos, a diminuição do preço dos combustíveis"".","Circula nas redes sociais uma publicação dizendo que, em 2022, oTribunal Superior Eleitoral(TSE) proibiu o governo do então presidenteJair Bolsonaro(PL) de reduzir os preços da gasolina em 2022.É#FAKE.",https://g1.globo.com/fato-ou-fake/noticia/2026/03/25/e-fake-que-tse-proibiu-bolsonaro-de-baixar-preco-de-combustiveis-em-2022.ghtml,FAKE
3,"Segundo o ex-ministro do TSE Carlos Horbach, relator do caso em 2022,a consulta da AGU foi rejeitada por falta de objetividade. No voto, ele afirmou:""É inviável a formulação de consultas para análises de possíveis condutas vedadas, uma vez que sua verificação exige minuciosa análise das circunstâncias fáticas concretas. À luz do entendimento do TSE, a abstração se traduz na completa desvinculação de casos concretos, o que deve ser aliado à necessária objetividade do questionamento, sob pena do cabimento de inúmeras respostas possíveis"".","Circula nas redes sociais uma publicação dizendo que, em 2022, oTribunal Superior Eleitoral(TSE) proibiu o governo do então presidenteJair Bolsonaro(PL) de reduzir os preços da gasolina em 2022.É#FAKE.",https://g1.globo.com/fato-ou-fake/noticia/2026/03/25/e-fake-que-tse-proibiu-bolsonaro-de-baixar-preco-de-combustiveis-em-2022.ghtml,FAKE
4,"Não há, no texto do ECA Digital, qualquer menção ao bloqueio de aparelhos eletrônicos ou de sistemas operacionais como um todo. É prevista, sim, a implementação de mecanismos de verificação de idade para impedir o acesso de crianças e adolescentes a aplicativos ou plataformas que apresentem conteúdo digital inadequado, mas uma possível 'barragem' não acarretaria na inutilização do aparelho usado.","Circulam nas redes sociais publicações afirmando que iPhones poderiam ser bloqueados no Brasil após a entrada em vigor da lei 15.211/2025,batizada de Estatuto da Criança e do Adolescente Digital (ECA Digital)e apelidada de “Lei Felca”, em referência ao influenciador Felipe Bressanim,que

In [117]:
extracted_df['status'].value_counts()

status
FAKE            2046
FATO            1118
NAOEBEMASSIM     587
Name: count, dtype: int64

In [118]:
extracted_df[extracted_df['status'] == 'FAKE'][['blockquote_text', 'next_item_text', 'url', 'status']].sample(20)

,blockquote_text,next_item_text,url,status
625,"""Os falsos anúncios atraem os pacientes oferecendo medicamentos mais baratos e até de forma gratuita 'após a realização de cadastro'. Atenção: os anúncios são falsos. A Anvisa não comercializa qualquer medicamento ou serve de intermediária para a sua venda"", detalha o comunicado.","Circulam nas redes sociais anúncios que simulam símbolos do governo federal e levam a site que imita a página da Agência Nacional de Vigilância Sanitária (Anvisa) com a promessa de venda de Mounjaro (uma das ""canetas"" usadas no tratamento de obesidade e diabetes).É#FAKE.",https://g1.globo.com/fato-ou-fake/noticia/2025/10/09/e-fake-site-que-imita-pagina-da-anvisa-e-promete-venda-de-mounjaro.ghtml,FAKE
494,"O resultado levou a um vídeo publicado no próprio dia da megaoperação, 28 de outubro, pelo perfil oficial de Cláudio Castro no Instagram. Na gravação, é possível ver que o governador faz os mesmos gestos visto no material falso e usa as mesmas roupas. Porém, em nenhum momento, menciona Lula ou o processo de qual é acusado. Em vez disso, faz uma atualização sobre os desdobramentos da ação policial .","Circula nas redes sociais uma publicação que mostra o governador doRio de Janeiro,Cláudio Castro(PL), dizendo que a esquerda e o presidenteLula(PT) irão cassá-lo.É #FAKE.",https://g1.globo.com/fato-ou-fake/noticia/2025/11/11/e-fake-video-de-claudio-castro-dizendo-que-lula-e-esquerda-vao-cassa-lo-cena-foi-manipulada-com-inteligencia-artificial.ghtml,FAKE
1502,"""Essa medida do Banco Central tem o objetivo de combater fraudes, uma vez que criminosos frequentemente utilizam CPFs de pessoas falecidas ou CNPJs irregulares para aplicar golpes. Portanto, ter impostos em atraso não impede ninguém de utilizar o Pix.""",Circulam nas redes sociais publicações dizendo que o governo Lula vai excluir o PIX de quem deve impostos àReceita Federal.É #FAKE.,https://g1.globo.com/fato-ou-fake/noticia/2025/03/13/e-fake-que-o-governo-vai-excluir-o-pix-de-quem-deve-imposto-a-receita-federal.ghtml,FAKE
2292,“Que rachadinha? Rachadinha ‘é seus’ filhos roubando milhões de empresas após a sua chegada no poder.”,"A declaração é #FAKE. Veja por quê:Segundo dados do Departamento Intersindical de Estatística e Estudos Socioeconômicos (Dieese) o salário mínimo em janeiro de 2003, quando Lula assumiu, era de R$ 200. No último ano de governo do petista, em 2010,o salário mínimo era de R$ 510. Considerando a inflação, o aumento real é de 54%.",https://g1.globo.com/fato-ou-fake/politica/noticia/2022/09/30/veja-o-que-e-fato-ou-fake-nas-falas-dos-presidenciaveis-no-debate-da-globo.ghtml,FAKE
791,"""O texto no documento em poder da pessoa parece incomumente claro e imposto digitalmente, especialmente na página direita. O estilo da fonte é inconsistente com o de documentos oficiais típicos"".","Circula nas redes sociais uma imagem que mostra o presidente dosEstados Unidos,Donald Trump, mostrandoa suposta carta ao bilionário Jeffrey Epstein na qual aparece o desenho de uma mulher nua.É #FAKE.",https://g1.globo.com/fato-ou-fake/noticia/2025/09/09/e-fake-foto-de-trump-mostrando-suposta-carta-a-epstein-com-desenho-de-mulher-nua-trata-se-de-montagem.ghtml,FAKE
3533,"“Quando fui governador, autorizei o uso das bicicletas nas rodovias paulistas, inclusive criando a ‘Rota Márcia Prado’, na descida da serra, que foi um sucesso” (no Twitter, em 25/10)","A declaração é #FAKE. Veja o porquê:Ao contrário do declarado pelo candidato, a atual gestão, iniciada em janeiro de 2017, requalificou, ou seja, reformou, 28,46 km de corredores de ônibus, número bem abaixo do número citado por Bruno Covas (PSDB), segundo a própria assessoria dele.",https://g1.globo.com/fato-ou-fake/noticia/2020/10/31/veja-o-que-e-fato-ou-fake-nas-declaracoes-dos-candidatos-a-prefeitura-de-sp-na-5a-semana-de-campanha.ghtml,FAKE
3097,"""Quanto eu gastei na minha campanha? Eu gastei R$ 2,5 milhões. Em 2018.""","A declaração é #FAKE. Veja por quê:A PEC do teto de gastosfoi promulgada no

In [120]:
extracted_df[extracted_df['next_item_text'].str.contains('porque')][['blockquote_text', 'next_item_text', 'url', 'status']].sample(20)

,blockquote_text,next_item_text,url,status
368,"""Por um erro de digitação -- provocado e reconhecido documentalmente pela Funai -- a demarcação da Terra Indígena Uru-Eu-Wau-Wau acabou se sobrepondo à área do assentamento. O Marco 26, descrito no projeto da Terra Indígena, está localizado na nascente do rio Igarapé Norte-Sul; entretanto, as coordenadas da nascente foram inseridas de forma incorreta, a quilômetros de distância, invadindo a área dos produtores"".",Circula nas redes sociais a publicação dizendo que oIbamaexpulsou famílias de uma área rural em Rondônia porque o governo teria vendido as terras para a China.É #FAKE.,https://g1.globo.com/fato-ou-fake/noticia/2025/12/15/e-fake-que-video-mostre-ibama-expulsando-familias-de-suas-terras-e-que-territorio-tenha-sido-vendido-para-a-china.ghtml,FAKE
3271,"""Em 7 de janeiro, o governo federal contratou 46 milhões de doses de vacinas do Butantan e, em fevereiro, mais 54 milhões de doses, além de 10 milhões de doses da União Química e 20 milhões da Precisa Medicamentos, com previsão de mais 30 milhões de doses do Butantan. Em março de 2021, mais 100 milhões de doses da Pfizer e 38 milhões de doses da Janssen”","A declaração é #FAKE. Veja o porquê:Houve, sim, uma pausa nas negociações com o instituto para a aquisição da Coronavac. Em outubro, o então ministro Eduardo Pazuello anunciou um protocolo de intenção de compra de 46 milhões de doses do imunizante, produzido pelo Instituto Butantan em parceria com a Sinovac. Um dia depois, Bolsonaro afirmou que o governo federal não ia adquirir ""vacina da China"":""Ele [Pazuello] tem um protocolo de intenções, já mandei cancelar se ele assinou. Já mandei cancelar. O presidente sou eu, não abro mão da minha autoridade. Até porque estaria comprando uma vacina que ninguém está interessado por ela, a não ser nós"".",https://g1.globo.com/fato-ou-fake/noticia/2021/06/09/veja-o-que-e-fato-ou-fake-nas-declaracoes-do-ex-secretario-elcio-franco-na-cpi-da-covid.ghtml,FAKE
366,"E prossegue: ""A desintrusão atende decisão do Supremo Tribunal Federal, no âmbito daADPF 709e tem como missão preservar a sustentabilidade, integridade e cultura dos povos indígenas, além de garantir a integridade do território e a proteção do meio ambiente da região"".",Circula nas redes sociais a publicação dizendo que oIbamaexpulsou famílias de uma área rural em Rondônia porque o governo teria vendido as terras para a China.É #FAKE.,https://g1.globo.com/fato-ou-fake/noticia/2025/12/15/e-fake-que-video-mostre-ibama-expulsando-familias-de-suas-terras-e-que-territorio-tenha-sido-vendido-para-a-china.ghtml,FAKE
637,"OFato ou Fakemostrou a publicação ao professor Thiago Carita Correra, do Departamento de Química Fundamental do Instituto de Química da Universidade de São Paulo (USP). Ele afirma: ""Não é verdade. O metanol não causa alteração de cor. Não existe teste para metanol que possa ser feito pelo consumidor. Apenas em laboratório""(leia detalhes abaixo).","Circula nas redes sociais o vídeo de um ""teste caseiro"" para, supostamente, detectar se uma bebida alcoólica tem metanol. A cena mostra uma pessoa despejando um líquido em uma fatia de pão de forma, e a descrição diz que ""se fica preto, é porque tem [a] substância"".É #FAKE.",https://g1.globo.com/fato-ou-fake/noticia/2025/10/10/e-fake-que-teste-com-pao-detecta-se-ha-metanol-na-bebida-alcoolica.ghtml,FAKE
367,"""A destruição das estruturas ocorreu após conferência dos bancos de dados, análise de imagens e checagem dos marcos geográficos in loco que confirmaram que a região está dentro dos limites da TI. Os ocupantes das duas fazendas não possuíam qualquer documentação válida ou legal de posse da área. Não há no local Programas de Assentamentos do Incra ou qualquer outro tipo de política de distribuição de terras"".",Circula nas redes sociais a publicação dizendo que oIbamaexpulsou famílias de uma área rural em Rondônia porque o governo teria vendido as terras para a China.É #FAKE.,https://g1.globo.com/fato-ou-fake/n

In [121]:
final_df = extracted_df[['blockquote_text', 'next_item_text', 'url', 'status']].copy()

final_df.head()

,blockquote_text,next_item_text,url,status
0,"O vídeo foi gravado no cemitério de Annotto Bay, na região nordeste do país. Segundo uma reportagem do jornal local ""Jamaica Observer"", os totens se tornaram uma tradição popular por lá, para auxiliar na localização dos túmulos e manter viva a memória dos falecidos. Um funcionário explicou, em entrevista, que os próprios familiares das pessoas enterradas escolhem as fotos, que depois são transformadas em grandes imagens em 2D. Para fixá-las, uma haste de aço é cimentada na sepultura.","Circula nas redes sociais um vídeo que mostra um cemitério na Jamaica repleto de totens de papelão, confeccionados em tamanho real, para representar as pessoas enterradas no local.É #FATO.",https://g1.globo.com/fato-ou-fake/noticia/2026/03/23/e-fato-cemiterio-na-jamaica-tem-totens-com-fotos-em-tamanho-real-de-pessoas-enterradas.ghtml,FATO
1,OFato ou Fakepesquisou pelo cemitério de Annotto Bay no Google Maps e confirmou que os totens são visíveis a partir de ruas ao lado do cemitério.Veja as imagens disponíveis no Google Street View:,"Circula nas redes sociais um vídeo que mostra um cemitério na Jamaica repleto de totens de papelão, confeccionados em tamanho real, para representar as pessoas enterradas no local.É #FATO.",https://g1.globo.com/fato-ou-fake/noticia/2026/03/23/e-fato-cemiterio-na-jamaica-tem-totens-com-fotos-em-tamanho-real-de-pessoas-enterradas.ghtml,FATO
2,"Mas isso não é verdade.OFato ou Fakeenviou o material à assessoria de imprensa do TSE, que respondeu, por e-mail:""O conteúdo é falso. Em 22 de março de 2022, o TSE arquivou consulta formulada [em 16 de fevereiro] pelaAdvocacia-Geral da União(AGU) que questionava se é possível, em ano eleitoral, a redução, a partir de lei aprovada pelo Congresso Nacional, da alíquota de impostos e contribuições sobre produtos e insumos. Por unanimidade, o Plenário decidiu não conhecer da consulta, o que significa que o julgamento não foi levado adiante para análise do mérito da questão, que envolve, entre outros pontos, a diminuição do preço dos combustíveis"".","Circula nas redes sociais uma publicação dizendo que, em 2022, oTribunal Superior Eleitoral(TSE) proibiu o governo do então presidenteJair Bolsonaro(PL) de reduzir os preços da gasolina em 2022.É#FAKE.",https://g1.globo.com/fato-ou-fake/noticia/2026/03/25/e-fake-que-tse-proibiu-bolsonaro-de-baixar-preco-de-combustiveis-em-2022.ghtml,FAKE
3,"Segundo o ex-ministro do TSE Carlos Horbach, relator do caso em 2022,a consulta da AGU foi rejeitada por falta de objetividade. No voto, ele afirmou:""É inviável a formulação de consultas para análises de possíveis condutas vedadas, uma vez que sua verificação exige minuciosa análise das circunstâncias fáticas concretas. À luz do entendimento do TSE, a abstração se traduz na completa desvinculação de casos concretos, o que deve ser aliado à necessária objetividade do questionamento, sob pena do cabimento de inúmeras respostas possíveis"".","Circula nas redes sociais uma publicação dizendo que, em 2022, oTribunal Superior Eleitoral(TSE) proibiu o governo do então presidenteJair Bolsonaro(PL) de reduzir os preços da gasolina em 2022.É#FAKE.",https://g1.globo.com/fato-ou-fake/noticia/2026/03/25/e-fake-que-tse-proibiu-bolsonaro-de-baixar-preco-de-combustiveis-em-2022.ghtml,FAKE
4,"Não há, no texto do ECA Digital, qualquer menção ao bloqueio de aparelhos eletrônicos ou de sistemas operacionais como um todo. É prevista, sim, a implementação de mecanismos de verificação de idade para impedir o acesso de crianças e adolescentes a aplicativos ou plataformas que apresentem conteúdo digital inadequado, mas uma possível 'barragem' não acarretaria na inutilização do aparelho usado.","Circulam nas redes sociais publicações afirmando que iPhones poderiam ser bloqueados no Brasil após a entrada em vigor da lei 15.211/2025,batizada de Estatuto da Criança e do Adolescente Digital (ECA Digital)e apelidada de “Lei Felca”, em referência ao influenciador Felipe Bressanim,que

In [123]:
save = True

if save:
    final_df.to_csv('out/20260326_g1_extracted_blockquotes_reasonings.csv', index=False, sep='|')